# mini_sim_db tutorial

This notebook explains the full idea behind `mini_sim_db` and shows practical usage beyond the short README.

## 1) Design idea

`mini_sim_db` is intentionally simple:

- Store simulation run metadata in a CSV file (human-readable, easy to diff, easy to back up).
- Keep a tiny local CLI for direct use in submit/post scripts.
- Offer a small REST layer for centralized/team workflows.
- Reuse one core CSV logic path (`sim_db.py`) from both CLI and server/client code.

This avoids heavyweight databases while still giving structured run tracking.

## 2) Schema and fields

The CSV keeps these fields in stable order:

- `case`: case ID (primary key style identifier)
- `work_dir`: run working directory
- `bin`: solver/executable label
- `inp`: primary input file
- `input_files`: all input files joined with `;`
- `extra_params`: extra runtime parameters (usually JSON text)
- `run_host`: host that initiated the mutating operation (mainly set by REST client)
- `status`: `start`, `restart`, or `done`
- `note`: short human note
- `notes`: legacy compatibility field
- `state_changed_at`: timestamp for status lifecycle events
- `created_at`: initial creation timestamp
- `updated_at`: last update timestamp

### `extra_params` in practice

You can store runtime knobs that are not represented as dedicated columns, for example:

```json
{"threads": 16, "precision": "double", "queue": "long"}
```

## 3) Local-only workflow (CLI)

Use this when each machine/user writes its own CSV.

In [ ]:
# init
python sim_db.py init

# add a case with extra params
python sim_db.py add \
  --case wing_040 \
  --work-dir /scratch/wing_040 \
  --inp wing_040.inp \
  --bin solver_v2 \
  --extra-param threads=16 \
  --extra-param precision=double \
  --status start

# done + list
python sim_db.py done --case wing_040
python sim_db.py list

## 4) Remote workflow (REST server + client)

Use this when you want one central DB and access from many machines.

In [ ]:
# terminal A: server
export SIM_DB_API_TOKEN='replace-me'
python sim_db_server.py --host 127.0.0.1 --port 8765 --db ~/sim_db.csv

In [ ]:
# terminal B: client
export SIM_DB_API_TOKEN='replace-me'
python sim_db_client.py --url http://127.0.0.1:8765 init
python sim_db_client.py --url http://127.0.0.1:8765 create \
  --case c100 --inp c100.inp --bin solver --status start \
  --extra-params '{"threads":8,"mode":"fast"}'
python sim_db_client.py --url http://127.0.0.1:8765 done --case c100
python sim_db_client.py --url http://127.0.0.1:8765 list

### Local durability + `run_host`

By default, mutating client operations use dual-write behavior:

1. write to local mirror CSV (`~/.sim_db_client_local.csv`)
2. attempt remote write

If remote is temporarily down, operation can still be kept locally.

`run_host` is set from the client side to record which machine initiated a write.

## 5) Security notes

Keep this lightweight but not open-by-default:

- Always set and require `SIM_DB_API_TOKEN` (or pass `--token`).
- Bind to loopback (`127.0.0.1`) unless you explicitly need remote access.
- If exposing remotely, use firewall/Tailscale/VPN and strict path allowlists.
- Prefer environment variables for tokens over hardcoding them in scripts.
- Keep `--allowed-db-path` / `--allowed-base-dir` tight on server side.

## 6) Practical examples

### Example A: submit script integration

Record `start` before submission, then `done` after post-processing.

In [ ]:
#!/usr/bin/env bash
set -euo pipefail

CASE=case_123
DB=${DB:-$HOME/sim_db.csv}

python sim_db.py init --db "$DB"
python sim_db.py add --case "$CASE" --inp "${CASE}.inp" --bin solver --status start --db "$DB"
# ... submit workload ...
python sim_db.py done --case "$CASE" --db "$DB"

### Example B: track restarts clearly

Use `status=restart` with a short note so later analysis is easier.

In [ ]:
python sim_db.py add \
  --case case_777 \
  --inp case_777.inp \
  --bin solver_v3 \
  --status restart \
  --note 'rerun after mesh repair'

---

If you only need the quick commands, go back to `README.md`.
If you need structure and context, keep this notebook as your reference.